# 🎙️ Chatterbox TTS Fine-Tuning on Google Colab

This notebook provides a complete workflow for fine-tuning Chatterbox TTS models (Standard and Turbo) with LoRA support.

## Overview
- **Setup**: Install dependencies and clone repository
- **Configuration**: Choose between Standard/Turbo mode and LoRA/Full Fine-Tune
- **Dataset Preparation**: Upload your dataset or use sample data
- **Training**: Fine-tune the model with your custom voice
- **Inference**: Generate speech with your trained model

### ⚠️ Important Notes
- **GPU Runtime Required**: Go to `Runtime > Change runtime type > GPU`
- **LoRA Recommended**: For datasets < 10 hours, use LoRA (`is_lora = True`)
- **Turbo Mode**: Faster training but may require LoRA for stability


## Step 1: Clone Repository & Install Dependencies

In [ ]:
#@title 📦 Clone Repository and Install Dependencies
#@markdown This cell clones the repository and installs all required packages including FFmpeg.

import os
import subprocess

# Navigate to workspace
%cd /workspace

# Install system dependencies (FFmpeg)
print("Installing FFmpeg...")
!apt-get update -qq
!apt-get install -y ffmpeg -qq

# Install Python dependencies
print("Installing Python dependencies...")
!pip install -q -r requirements.txt

# Verify installation
print("\n✅ Installation complete!")
print(f"Python version: {subprocess.check_output(['python', '--version']).decode().strip()}")
print(f"FFmpeg version: {subprocess.check_output(['ffmpeg', '-version']).decode().split()[2]}")

# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## Step 2: Configure Training Settings

In [ ]:
#@title ⚙️ Configure Training Settings
#@markdown Set your training preferences here before running setup.py

#@markdown ### Model Selection
IS_TURBO = True #@param {type:"boolean"}
#@markdown - **Turbo** (`True`): GPT-2 based, faster training, better for English base
#@markdown - **Standard** (`False`): Llama-based, character-level tokenizer

#@markdown ### Training Strategy
IS_LORA = True #@param {type:"boolean"}
#@markdown - **LoRA** (`True`): Efficient, less VRAM, recommended for <10h datasets
#@markdown - **Full Fine-Tune** (`False`): Trains all parameters, needs more VRAM

#@markdown ### Dataset Format
DATASET_FORMAT = "LJSpeech" #@param ["LJSpeech", "FileBased", "JSON"]
#@markdown - **LJSpeech**: metadata.csv format (filename|raw_text|normalized_text)
#@markdown - **FileBased**: Individual .txt and .wav pairs
#@markdown - **JSON**: JSON formatted dataset

# Update config file
config_path = "/workspace/src/config.py"

with open(config_path, 'r') as f:
    config_content = f.read()

# Update settings
config_content = config_content.replace(
    "is_turbo: bool = False",
    f"is_turbo: bool = {str(IS_TURBO).lower()}"
)
config_content = config_content.replace(
    "is_lora: bool = True",
    f"is_lora: bool = {str(IS_LORA).lower()}"
)

# Update dataset format
if DATASET_FORMAT == "LJSpeech":
    config_content = config_content.replace(
        "ljspeech = True",
        "ljspeech = True"
    )
    config_content = config_content.replace(
        "json_format = False",
        "json_format = False"
    )
elif DATASET_FORMAT == "FileBased":
    config_content = config_content.replace(
        "ljspeech = True",
        "ljspeech = False"
    )
    config_content = config_content.replace(
        "json_format = False",
        "json_format = False"
    )
elif DATASET_FORMAT == "JSON":
    config_content = config_content.replace(
        "ljspeech = True",
        "ljspeech = False"
    )
    config_content = config_content.replace(
        "json_format = False",
        "json_format = True"
    )

with open(config_path, 'w') as f:
    f.write(config_content)

print(f"✅ Configuration updated:")
print(f"   - is_turbo: {IS_TURBO}")
print(f"   - is_lora: {IS_LORA}")
print(f"   - dataset_format: {DATASET_FORMAT}")

if IS_TURBO:
    print("\n⚠️ NOTE: After running setup.py, you'll need to update new_vocab_size in config.py")
    print("   The setup script will provide the exact value to use.")

## Step 3: Download & Setup Base Models

In [ ]:
#@title 📥 Download Base Models (Run setup.py)
#@markdown This downloads the required base models (ve, s3gen, t3) and tokenizer.
#@markdown **Important**: If using Turbo mode, note the vocab_size value outputted below!

%cd /workspace

# Clean previous models if switching modes
if os.path.exists("./pretrained_models"):
    user_input = input("Pre-existing models found. Delete and re-download? (y/n): ")
    if user_input.lower() == 'y':
        !rm -rf ./pretrained_models
        print("Deleted pretrained_models directory.")

# Run setup
print("Running setup.py...\n")
!python setup.py

print("\n" + "="*50)
print("✅ Setup complete!")
print("="*50)

## Step 4: Update Vocabulary Size (Turbo Mode Only)

In [ ]:
#@title 🔢 Update Vocabulary Size (Turbo Mode ONLY)
#@markdown If you selected **Turbo mode**, enter the vocab_size value shown by setup.py above.
#@markdown For Standard mode, leave it at 2454.

VOCAB_SIZE = 52260 #@param {type:"integer"}
#@markdown Enter the vocab_size value from setup.py output (e.g., 52260 for Turbo, 2454 for Standard)

config_path = "/workspace/src/config.py"

with open(config_path, 'r') as f:
    config_content = f.read()

# Find and replace the new_vocab_size line
import re
config_content = re.sub(
    r'new_vocab_size: int = \d+',
    f'new_vocab_size: int = {VOCAB_SIZE}',
    config_content
)

with open(config_path, 'w') as f:
    f.write(config_content)

print(f"✅ Updated new_vocab_size to {VOCAB_SIZE}")

# Verify
with open(config_path, 'r') as f:
    for line in f:
        if 'new_vocab_size' in line:
            print(f"   {line.strip()}")

## Step 5: Upload Your Dataset

In [ ]:
#@title 📁 Upload Your Dataset
#@markdown Choose your upload method based on dataset format.

from google.colab import files
import zipfile
import os

UPLOAD_METHOD = "Manual Upload" #@param ["Manual Upload", "Use Sample Dataset", "Google Drive"]

if UPLOAD_METHOD == "Manual Upload":
    print("📤 Upload your dataset ZIP file...")
    print("Expected format:")
    print("  - LJSpeech: metadata.csv + wavs/ folder")
    print("  - FileBased: .txt and .wav pairs")
    print("  - JSON: metadata.json + audio files")
    
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            print(f"\nExtracting {filename}...")
            # Extract to appropriate directory based on content
            !unzip -o {filename} -d /workspace/
            print(f"✅ Extracted {filename}")
        else:
            print(f"⚠️ Please upload a ZIP file containing your dataset.")

elif UPLOAD_METHOD == "Use Sample Dataset":
    print("Using the sample dataset already in the repository...")
    # Check if sample dataset exists
    if os.path.exists("/workspace/MyTTSDataset/metadata.csv"):
        print("✅ Sample dataset found at /workspace/MyTTSDataset/")
        !ls -la /workspace/MyTTSDataset/
    else:
        print("⚠️ Sample dataset not found. Please upload your own.")

elif UPLOAD_METHOD == "Google Drive":
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    
    # Ask for dataset path
    dataset_path = input("Enter the path to your dataset in Google Drive: ")
    
    # Copy to workspace
    if os.path.exists(dataset_path):
        !cp -r {dataset_path}/* /workspace/MyTTSDataset/
        print(f"✅ Copied dataset from {dataset_path}")
    else:
        print(f"⚠️ Path not found: {dataset_path}")

# Verify dataset
print("\n" + "="*50)
print("Dataset Verification:")
print("="*50)
if os.path.exists("/workspace/MyTTSDataset/metadata.csv"):
    print("✅ LJSpeech format detected")
    !head -5 /workspace/MyTTSDataset/metadata.csv
elif os.path.exists("/workspace/FileBasedDataset"):
    print("✅ FileBased format detected")
    !ls /workspace/FileBasedDataset/ | head -10
else:
    print("⚠️ No dataset found. Please upload your dataset.")

## Step 6: Adjust Training Hyperparameters (Optional)

In [ ]:
#@title 🎛️ Adjust Training Hyperparameters
#@markdown Fine-tune these settings based on your dataset size and GPU capabilities.

#@markdown ### Batch Size (adjust for VRAM)
BATCH_SIZE = 16 #@param {type:"slider", min:2, max:64, step:2}
#@markdown Higher = faster training but more VRAM. Reduce if you get OOM errors.

#@markdown ### Learning Rate
LEARNING_RATE = 0.0001 #@param {type:"number"}
#@markdown Recommended: 1e-4 for LoRA, 1e-5 for Full Fine-Tune

#@markdown ### Number of Epochs
NUM_EPOCHS = 10 #@param {type:"integer"}
#@markdown More epochs = better adaptation but risk of overfitting

#@markdown ### Save Steps
SAVE_STEPS = 200 #@param {type:"integer"}
#@markdown How often to save checkpoints during training

# Update config
config_path = "/workspace/src/config.py"

with open(config_path, 'r') as f:
    config_content = f.read()

import re
config_content = re.sub(r'batch_size: int = \d+', f'batch_size: int = {BATCH_SIZE}', config_content)
config_content = re.sub(r'learning_rate: float = [\d.eE+-]+', f'learning_rate: float = {LEARNING_RATE}', config_content)
config_content = re.sub(r'num_epochs: int = \d+', f'num_epochs: int = {NUM_EPOCHS}', config_content)
config_content = re.sub(r'save_steps: int = \d+', f'save_steps: int = {SAVE_STEPS}', config_content)

with open(config_path, 'w') as f:
    f.write(config_content)

print(f"✅ Updated hyperparameters:")
print(f"   - batch_size: {BATCH_SIZE}")
print(f"   - learning_rate: {LEARNING_RATE}")
print(f"   - num_epochs: {NUM_EPOCHS}")
print(f"   - save_steps: {SAVE_STEPS}")

# Show current config summary
print("\n" + "="*50)
print("Current Training Configuration:")
print("="*50)
!grep -E "(is_turbo|is_lora|batch_size|learning_rate|num_epochs|new_vocab_size)" /workspace/src/config.py

## Step 7: Start Training

In [ ]:
#@title 🚀 Start Training
#@markdown This will begin the fine-tuning process. Training time depends on dataset size and GPU.
#@markdown You can monitor progress and listen to inference samples during training.

%cd /workspace

# Check if dataset exists
if not os.path.exists("/workspace/MyTTSDataset/metadata.csv") and not os.path.exists("/workspace/FileBasedDataset"):
    print("❌ ERROR: No dataset found! Please upload your dataset in Step 5.")
else:
    print("Starting training...")
    print(f"Model: {'Turbo' if IS_TURBO else 'Standard'}")
    print(f"Training Method: {'LoRA' if IS_LORA else 'Full Fine-Tune'}")
    print(f"Expected Output: {'new_lang_adapter/' if IS_LORA else 't3_finetuned.safetensors'}")
    print("\n" + "="*50 + "\n")
    
    # Run training
    !python train.py
    
    print("\n" + "="*50)
    print("✅ Training Complete!")
    print("="*50)
    
    # Show output
    if IS_LORA:
        print("\nLoRA adapter saved to: /workspace/chatterbox_output/new_lang_adapter/")
        !ls -lh /workspace/chatterbox_output/new_lang_adapter/
    else:
        print("\nFine-tuned model saved to: /workspace/chatterbox_output/")
        !ls -lh /workspace/chatterbox_output/*.safetensors

## Step 8: Test Inference

In [ ]:
#@title 🗣️ Test Speech Synthesis (Inference)
#@markdown Generate speech with your trained model to test quality.

TEXT_TO_SPEAK = "Hello, this is a test of my newly trained text to speech model." #@param {type:"string"}
#@markdown Enter the text you want to synthesize

# Update inference.py with custom text
inference_path = "/workspace/inference.py"

with open(inference_path, 'r') as f:
    inference_content = f.read()

# Replace the test text
import re
inference_content = re.sub(
    r'TEXT_TO_SAY = "[^"]*"',
    f'TEXT_TO_SAY = "{TEXT_TO_SPEAK}"',
    inference_content
)

with open(inference_path, 'w') as f:
    f.write(inference_content)

print(f"Text to synthesize: {TEXT_TO_SPEAK}")
print("\nRunning inference...\n")

%cd /workspace
!python inference.py

# Play the output
if os.path.exists("/workspace/output.wav"):
    print("\n✅ Generated output.wav")
    from IPython.display import Audio, display
    display(Audio("/workspace/output.wav"))
else:
    print("\n⚠️ Output file not generated. Check training logs for errors.")

## Step 9: Merge LoRA Adapter (Optional)

In [ ]:
#@title 📦 Merge LoRA Adapter into Base Model
#@markdown If you trained with LoRA, this creates a standalone model file for deployment.
#@markdown Skip this step if you used Full Fine-Tune.

if IS_LORA:
    print("Merging LoRA adapter with base model...")
    %cd /workspace
    !python merge_lora.py
    
    print("\n✅ Merge complete!")
    print("Merged model saved to: /workspace/chatterbox_output/t3_turbo_finetuned_merged.safetensors")
    
    # Show file size
    if os.path.exists("/workspace/chatterbox_output/t3_turbo_finetuned_merged.safetensors"):
        !ls -lh /workspace/chatterbox_output/*.safetensors
else:
    print("⚠️ Skipping merge step (not needed for Full Fine-Tune)")
    print("Your model is already at: /workspace/chatterbox_output/t3_finetuned.safetensors")

## Step 10: Download Your Trained Model

In [ ]:
#@title 💾 Download Trained Model
#@markdown Download your fine-tuned model for local use or sharing.

from google.colab import files
import os

MODEL_TYPE = "LoRA Adapter" if IS_LORA else "Full Fine-Tuned Model"
print(f"Downloading: {MODEL_TYPE}")

if IS_LORA:
    # Zip the adapter folder
    !zip -r /workspace/chatterbox_output/new_lang_adapter.zip /workspace/chatterbox_output/new_lang_adapter/
    files.download("/workspace/chatterbox_output/new_lang_adapter.zip")
    print("\n✅ Downloaded LoRA adapter (zip file)")
    print("To use: Extract the zip and point inference.py to the new_lang_adapter/ folder")
else:
    # Download the safetensors file
    model_files = [f for f in os.listdir("/workspace/chatterbox_output/") if f.endswith('.safetensors')]
    if model_files:
        for model_file in model_files:
            files.download(f"/workspace/chatterbox_output/{model_file}")
            print(f"\n✅ Downloaded {model_file}")
    else:
        print("\n⚠️ No model files found. Make sure training completed successfully.")

print("\n" + "="*50)
print("🎉 Congratulations! Your Chatterbox TTS model is ready!")
print("="*50)

## Troubleshooting & Tips

In [ ]:
#@title 🛠️ Troubleshooting Utilities
#@markdown Useful commands for debugging common issues.

ACTION = "Check GPU Memory" #@param ["Check GPU Memory", "Check Disk Space", "List Output Files", "Clear Cache"]

if ACTION == "Check GPU Memory":
    print("GPU Memory Usage:")
    !nvidia-smi
    
elif ACTION == "Check Disk Space":
    print("Disk Space Usage:")
    !df -h /workspace
    
elif ACTION == "List Output Files":
    print("Output Directory Contents:")
    !ls -lh /workspace/chatterbox_output/
    
elif ACTION == "Clear Cache":
    print("Clearing Python cache...")
    !find /workspace -type d -name __pycache__ -exec rm -rf {} +
    !find /workspace -name "*.pyc" -delete
    print("✅ Cache cleared")